# 1 Probabilistic forecasting: the line is not the answer

Every model in `00_forecasting_baselines` output one number per half-hour. Suppose you hand
that to the person scheduling tomorrow's grid dispatch and they ask the question they
actually have:

> *What is the chance demand tops 9000 MW during tomorrow evening's peak?*

You cannot answer. Not because your model is bad — because a point forecast is not
the kind of object that has an answer to that question in it. It is a single line
through a future that has a width.

This notebook gives forecasts a width. It does it in the order that makes the ideas
stick rather than the order the papers were published:

1. a **free** probabilistic baseline, so the trained ones have something to beat;
2. **CRPS**, the metric that scores a distribution — and quietly generalises the MAE
   you already trust;
3. two heads you write yourself: a **Gaussian** head trained with NLL, and a
   **quantile** head trained with pinball loss;
4. **calibration** — is an 80% interval right 80% of the time? (Spoiler: not for every model, and not equally for every seed.)

The series and the windows are those of `00_forecasting_baselines`:
`australian_electricity_demand`, half-hourly grid demand with a clean daily-and-weekly
pattern, four days of history and one day of horizon, scaled on the training statistics.
That notebook named seasonal naive as the point baseline and trained a linear model and a
GRU on these windows; the heads below sit on the same two trunks. Calibration is easiest to
learn to read on a series that behaves.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.training import Trainer, TrainerSettings

from australian_electricity import load_ausalia
from windowing import window_index
from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")

DAY = 48  # half-hourly steps per day
# four days of history, one day of horizon: the window of 00_forecasting_baselines
W, H = 4 * DAY, DAY
STRIDE = 24  # 12 hours between training windows (00_forecasting_baselines 0.1)
# early stopping on the validation loss picks the epoch; this is only the ceiling
MAX_EPOCHS = param(100, test=2)
# the validation loss of one epoch wobbles; five epochs in a row without a new best
# is a stall, not a wobble (the patience of 00_forecasting_baselines)
PATIENCE = 5
# every trained row is three runs, so a gap between rows can be read against the spread
SEEDS = param((0, 1, 2), test=(0,))
torch.manual_seed(0)

ausalia = load_ausalia()
mu = ausalia.values[ausalia.splits["train"]].mean()
sigma = ausalia.values[ausalia.splits["train"]].std()


def windows(split: str, stride: int):
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", STRIDE * 2)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")

## 1.1 The width is real, and it is not your model's fault

Before blaming a model for missing the future, check how much of the future was
knowable. Take a window, find the other windows in the training set whose last 24
hours look almost identical to it, and plot what happened next in each case.

In [ ]:
probe = 400
recent = xtr[:, -DAY:]
distance = (recent - xtr[probe, -DAY:]).pow(2).mean(1)
neighbours = distance.argsort()[1:41]  # [0] is the probe itself

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(
    range(-DAY, 0), xtr[probe, -DAY:], color="k", lw=2, label="the last day we observed"
)
for j in neighbours:
    ax.plot(range(0, H), ytr[j], color="tab:blue", lw=0.7, alpha=0.35)
ax.plot(
    range(0, H), ytr[probe], color="crimson", lw=2, label="what actually happened once"
)
ax.axvline(0, color="0.8")
ax.set_xlabel("half-hours from the forecast origin")
ax.set_ylabel("scaled demand")
ax.set_title("40 windows whose last day matches, and the 40 futures that followed")
ax.legend(fontsize=8)
plt.tight_layout()

spread = torch.stack([ytr[j] for j in neighbours])
print(
    f"across these 40 windows, the outcome at +1 day ranges over "
    f"{float(spread[:, -1].max() - spread[:, -1].min()):.2f} scaled units"
)

Forty windows whose last day looks almost the same, and forty visibly different futures.
A model sees all four days of the window, so it could tell some of these apart: the fan is
not a measurement of what no model can know. It shows how much the last day alone leaves
open, and that this is a lot. Part of such a spread is **aleatoric** uncertainty,
irreducible given what you observe, as opposed to the **epistemic** kind you could reduce
with more data or a better model. This picture cannot separate the two; it only shows that
the width is there.

A point forecast reports the middle of that fan and says nothing about its width. The
rest of this notebook is about reporting the fan.

## 1.2 CRPS: one number that scores a distribution

Before building anything, decide how you will score it — otherwise you will pick the
model whose pictures you like.

The **Continuous Ranked Probability Score** works on a forecast expressed as *samples*.
Given `S` sample paths and the truth `y`:

$$\mathrm{CRPS} \approx \underbrace{\frac{1}{S}\sum_i |s_i - y|}_{\text{are the samples near the truth?}} - \underbrace{\frac{1}{2S(S-1)}\sum_{i \neq j} |s_i - s_j|}_{\text{are the samples spread out?}}$$

The first term rewards accuracy. The second is subtracted, so spread among the samples is
not punished twice: a forecast that is honestly uncertain is not scored as if each sample
were a separate miss. Widen the samples beyond the real uncertainty and the first term grows
faster than the second, so the score gets worse. Lower is better, and it is measured in
the units of the data.

The property that makes it the right metric to standardise on: **for a forecast with no
spread at all, CRPS is exactly MAE.** The second term vanishes. So CRPS is not a new
scale to learn — it is MAE, extended to forecasts that have a width, and every point
model in `00_forecasting_baselines` can be dropped straight into the same table.

In [ ]:
def crps(y: torch.Tensor, samples: torch.Tensor) -> float:
    # y: (N, H)   samples: (N, S, H)
    s = samples.shape[1]
    accuracy = (samples - y.unsqueeze(1)).abs().mean(1)
    spread = (samples.unsqueeze(1) - samples.unsqueeze(2)).abs().sum((1, 2)) / (
        2 * s * (s - 1)
    )
    return float((accuracy - spread).mean())


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def coverage(y: torch.Tensor, lo: torch.Tensor, hi: torch.Tensor) -> float:
    return float(((y >= lo) & (y <= hi)).float().mean())


# a "distribution" with no width: 100 copies of the same number
point = xva[:, -1:].expand(-1, H)
print(f"naive point forecast     MAE  {mae(yva, point):.4f}")
print(
    f"same forecast as samples CRPS {crps(yva, point.unsqueeze(1).expand(-1, 100, -1)):.4f}   <- identical"
)

## 1.3 The free baseline (naming it, again)

`00_forecasting_baselines` named its point baseline before training anything: on this
series that is seasonal naive, repeat yesterday (validation MAE 0.2906), with plain naive,
repeat the last half-hour, far behind (0.7362). Same rule here, and the probabilistic
baseline is almost as cheap as the point one.

Take the naive forecast — repeat the last value — and ask how wrong it was, historically,
at each horizon step. Those residuals *are* an empirical predictive distribution: resample
them and you have 100 sample paths, no training, no parameters.

In [ ]:
residuals = ytr - xtr[:, -1:]  # (N, H): how wrong "repeat the last value" was, on train

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(range(1, H + 1), residuals.std(0), marker="o", ms=3)
ax.set_xlabel("half-hours ahead")
ax.set_ylabel("residual std")
ax.set_title("how wrong repeating the last value was, per horizon step — measured")
plt.tight_layout()

draw = torch.randint(0, len(residuals), (len(xva), 100))
naive_samples = xva[:, -1:].unsqueeze(1) + residuals[draw]  # (N, 100, H)

scores: dict[str, dict] = {}


def score(samples: torch.Tensor, median: torch.Tensor) -> dict:
    lo, hi = samples.quantile(0.1, dim=1), samples.quantile(0.9, dim=1)
    return {
        "mae": mae(yva, median),
        "crps": crps(yva, samples),
        "cov80": coverage(yva, lo, hi),
    }


def mean_sd(xs: list[float]) -> tuple[float, float]:
    t = torch.tensor(xs, dtype=torch.float64)
    return float(t.mean()), float(t.std()) if len(xs) > 1 else 0.0  # sample std


def report(name: str, runs: list[dict], params: int) -> None:
    # runs: one dict per seed with "samples", "median" and, for a trained model, "fit"
    per_seed = [score(r["samples"], r["median"]) for r in runs]
    row = {"params": params, "fits": [r["fit"] for r in runs if "fit" in r]}
    for key in ("mae", "crps", "cov80"):
        row[key], row[key + "_sd"] = mean_sd([s[key] for s in per_seed])
    row["secs"] = mean_sd([f["secs"] for f in row["fits"]])[0] if row["fits"] else 0.0
    scores[name] = row
    print(
        f"{name:20s} MAE {row['mae']:.4f} +/- {row['mae_sd']:.4f}   "
        f"CRPS {row['crps']:.4f} +/- {row['crps_sd']:.4f}   "
        f"80% coverage {row['cov80']:.3f} +/- {row['cov80_sd']:.3f}   {params:>7,} params"
    )
    for seed, f in zip(SEEDS, row["fits"]):
        print(
            f"    seed {seed}: best epoch {f['best']:3d}, stopped after {f['ran']:3d} of {MAX_EPOCHS}"
        )


report(
    "naive + residuals",
    [{"samples": naive_samples, "median": naive_samples.median(1).values}],
    0,
)

Note what the picture already told us before any model was trained: the spread at
+1 half-hour is small and at +1 day is nearly **six times larger** (measured: 5.7x —
`residuals.std(0)[-1] / residuals.std(0)[0]`). Any forecast whose interval width is
constant across the horizon is wrong on its face.

## 1.4 A Gaussian head, trained with NLL

The smallest possible change to the GRU from `00_forecasting_baselines`: instead of one
number per horizon step, output **two** — a mean and a standard deviation — and train by maximum
likelihood instead of MSE.

$$-\log p(y \mid \mu, \sigma) = \frac{(y-\mu)^2}{2\sigma^2} + \log \sigma + \text{const}$$

Read the loss rather than just importing it. The first term is MSE **divided by the
variance the model chose** — so the model can reduce its loss on hard examples by
admitting they are hard. The `log σ` term is what stops it declaring everything hard:
uncertainty has to be paid for. Together they force an honest answer, and this is why
MSE is a special case (fix `σ = 1` and only the first term survives).

Two implementation details that are not optional. Predict `log σ` and exponentiate, so
`σ > 0` is structural rather than hoped for; and clamp it, because early in training an
unlucky batch can drive `σ` to zero and the gradient to infinity.

Nobody picks the number of epochs. `train` runs `vectormesh.training.Trainer` with early
stopping, as in lesson 1 (`1_representation_and_head/03_training` §3.5) and
`00_forecasting_baselines`: it watches the validation loss, stops after `PATIENCE = 5`
epochs without a new lowest value, and hands back the weights of the best epoch.
`MAX_EPOCHS` is only a ceiling. The loss needs the model and the target together, so it
goes in as a `step(model, x, y)` rather than a `loss_fn(yhat, y)`.

Two things to keep in mind when reading the table later. Each head stops on its own
training loss (the NLL here, the pinball loss for the quantile heads below), while the
table compares all of them on CRPS, MAE and coverage, which none of them was stopped on.
And because the validation split also chose the epoch, every trained score is a little
optimistic; the test year (E5) is where that is checked.

`fit_seeds` trains every head three times, with seeds 0, 1 and 2 (the seed fixes the
initial weights and the batch order), and every trained row reports the mean and the
sample standard deviation over those three runs, as `00_forecasting_baselines` does.

In [ ]:
class GRUTrunk(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.rnn = nn.GRU(1, hidden, batch_first=True)
        self.hidden = hidden

    def forward(self, x):
        out, _ = self.rnn(x.unsqueeze(-1))
        return out[:, -1, :]


class GaussianHead(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.trunk = GRUTrunk(hidden)
        self.mean = nn.Linear(hidden, H)
        self.log_sigma = nn.Linear(hidden, H)

    def forward(self, x):
        h = self.trunk(x)
        return self.mean(h), self.log_sigma(h).clamp(-7, 7).exp()


validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def train(model, step, batch_size=64, lr=1e-3) -> dict:
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    settings = TrainerSettings(
        epochs=MAX_EPOCHS,
        metrics=[],  # model(x) is a distribution here, not one yhat to score
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs={"patience": PATIENCE, "save": False, "verbose": False},
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=None,  # unused: `step` computes the loss
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        step=step,
        progress=False,
    )
    curve: list[float] = []  # the validation loss after every epoch
    trainer.reporters.append(lambda epoch, tr, te, metrics: curve.append(te))
    t0 = time.time()
    result = trainer.loop()  # trainer.model now holds the weights of the best epoch
    model.eval()
    return {
        "secs": time.time() - t0,
        "best": result.epoch + 1,  # epochs count from 0
        "ran": result.stopped_epoch + 1,
        "curve": curve,
    }


def fit_seeds(make_model, step) -> tuple[list[nn.Module], list[dict]]:
    models, fits = [], []
    for seed in SEEDS:
        torch.manual_seed(seed)  # the init and the batch order of this run
        model = make_model()
        fits.append(train(model, step))
        models.append(model)
    return models, fits


def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def gaussian_nll(model, x, y):
    mean, sd = model(x)
    return nn.functional.gaussian_nll_loss(mean, y, sd**2)


g_models, g_fits = fit_seeds(GaussianHead, gaussian_nll)
with torch.no_grad():
    g_out = [m(xva) for m in g_models]  # (mean, sigma) per seed, each (N, H)
g_runs = [
    {
        "samples": g_mu.unsqueeze(1)
        + g_sigma.unsqueeze(1) * torch.randn(len(g_mu), 100, H),
        "median": g_mu,
        "fit": fit,
    }
    for (g_mu, g_sigma), fit in zip(g_out, g_fits)
]
report("GRU + gaussian", g_runs, n_params(g_models[0]))

steps = {"+30min": 0, "+6h": 11, "+12h": 23, "+24h": 47}
g_sigma_mean = torch.stack([sd for _, sd in g_out]).mean(
    0
)  # (N, H), averaged over seeds
print("\n" + " " * 34 + "   ".join(f"{k:>6s}" for k in steps))
print(
    "learned sigma, mean over windows  "
    + "   ".join(f"{float(g_sigma_mean[:, j].mean()):6.3f}" for j in steps.values())
)
print(
    "naive residual std (1.3)          "
    + "   ".join(f"{float(residuals.std(0)[j]):6.3f}" for j in steps.values())
)
lo, hi = g_sigma_mean[:, -1].quantile(torch.tensor([0.1, 0.9]))
print(
    f"\nsigma at +24h across the {len(xva)} validation windows: 10th percentile {lo:.3f}, 90th {hi:.3f}"
)

Nobody told the model how wide its intervals should be at each horizon step; it learned
that from the data. The learned sigma is not a copy of the residual curve of 1.3. It rises
from 0.042 at +30 min to about 0.4 at +6 h and +12 h and falls to 0.184 at +24 h, while the
naive residuals are two to three times larger at every step shown. Those residuals measure
how wrong *repeating the last value* was; sigma is how wrong this model expects its own
mean to be. And sigma is set per *window*, not only per horizon step: at +24 h it ranges
from 0.156 to 0.213 between the 10th and the 90th percentile of the 361 validation windows.

As a point forecast, the mean of the Gaussian head has a validation MAE of 0.2508 ± 0.0166:
a third of the naive forecast's 0.7362 and ahead of seasonal naive (0.2906). The
MSE-trained GRU(64) of `00_forecasting_baselines` scored 0.2400 ± 0.0091 with the same
trunk and window. The difference, 0.011, is smaller than either spread, so these runs do
not show that predicting a width costs the mean anything.

## 1.5 But is it *calibrated*?

A model can produce beautiful intervals that are simply the wrong size. The check is
embarrassingly direct: an 80% interval should contain the truth 80% of the time. Count.

In [ ]:
levels = torch.tensor([0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
empirical = torch.zeros(len(SEEDS), len(levels))  # (seeds, levels)
for s, (g_mu, g_sigma) in enumerate(g_out):
    for k, level in enumerate(levels):
        z = torch.distributions.Normal(0, 1).icdf(0.5 + level / 2)
        empirical[s, k] = coverage(yva, g_mu - z * g_sigma, g_mu + z * g_sigma)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0.4, 1], [0.4, 1], "k--", lw=1, label="perfectly calibrated")
for s, seed in enumerate(SEEDS):
    ax.plot(
        levels, empirical[s], marker="o", ms=4, label=f"GRU + gaussian, seed {seed}"
    )
ax.set_xlabel("nominal coverage")
ax.set_ylabel("empirical coverage")
ax.set_title("analytic intervals, mean +/- z sigma")
ax.legend(fontsize=8)
plt.tight_layout()

print("nominal   " + "".join(f"seed {seed:<4d}" for seed in SEEDS) + "  mean")
for k, level in enumerate(levels):
    emp = float(empirical[:, k].mean())
    flag = (
        "too wide"
        if emp > level + 0.02
        else ("too narrow" if emp < level - 0.02 else "ok")
    )
    per_seed = "".join(f"{float(e):<9.1%} " for e in empirical[:, k])
    print(f"{float(level):<9.0%} {per_seed} {emp:.1%}   {flag}")

Averaged over the three seeds, all six levels sit inside the "ok" band of two points, about
what 361 validation windows can resolve; at 80% the analytic interval, `mean ± 1.28 sigma`,
covers 80.5% of the truth. The seeds do not agree with each other, though. Seed 0 is too
wide at every level (84.3% at 80%), seeds 1 and 2 a little too narrow (78.4% and 78.9%):
three runs of the same head differ by six points, more than their mean misses by. A
calibration curve describes one trained model, not the architecture. Check the model you
ship.

The table in 1.8 reports a different 80% number for the same head: `cov80 = 0.791`. Both
are right about what they measure. The curve here uses the analytic interval of the
predicted Gaussian; the table, like every row in it, takes the 10% and 90% quantiles of 100
sample paths. Between those two sample quantiles lies on average about 78% of the
distribution, not 80% (order statistics: 79.2 of 101 gaps), so the sampled interval is a
little narrower than the analytic one, and the 80.5% becomes 79.1%. That gap is the price
of scoring a distribution through samples. So the discipline is twofold: check the level
you actually report, and know which interval that number was computed from.

When an interval under-covers, as seeds 1 and 2 do slightly here, the model is
overconfident rather than cautious, and that is the less friendly failure: if somebody
sizes a safety margin off such an interval, they get less protection than the number
promised.

Calibration is a property you must measure separately from accuracy. CRPS folds the two
together into one number, which makes it a good ranking metric and a poor diagnostic —
so report both, always.

## 1.6 A quantile head: dropping the Gaussian assumption

Why should the future be symmetric and bell-shaped? Predict the quantiles directly and
find out. The tool is the **pinball loss** (also called quantile loss): for the level
`q`, penalise being too low with weight `q` and too high with weight `1 - q`.

$$L_q(y, \hat y) = \max\big(q\,(y - \hat y),\ (q - 1)(y - \hat y)\big)$$

At `q = 0.5` that is MAE, and its minimiser is the median. At `q = 0.9` under-prediction
costs nine times what over-prediction does, so the minimiser sits where 90% of the mass
is below it. Nine levels, nine outputs per horizon step, no distributional assumption
anywhere.

In [ ]:
QUANTILES = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])


def pinball(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # pred (B, Q, H), y (B, H)
    err = y.unsqueeze(1) - pred
    q = QUANTILES.view(1, -1, 1)
    return torch.maximum(q * err, (q - 1) * err).mean()


def sample_from_quantiles(pred: torch.Tensor, n: int = 100) -> torch.Tensor:
    # pred (N, Q, H), sorted along Q -> (N, n, H)
    # inverse-transform sampling: u ~ U(0, 1) read off the piecewise-linear quantile
    # function through the nine levels; below q10 and above q90 the outermost segment
    # is extended with its own slope, so the tails beyond the outer levels get samples too
    N, Q, _ = pred.shape
    u = torch.rand(N, n, H)
    t = (u - QUANTILES[0]) / (QUANTILES[1] - QUANTILES[0])  # position on the level grid
    idx = t.floor().long().clamp(0, Q - 2)
    grid = pred.unsqueeze(1).expand(-1, n, -1, -1)
    lo = torch.gather(grid, 2, idx.unsqueeze(2)).squeeze(2)
    hi = torch.gather(grid, 2, (idx + 1).unsqueeze(2)).squeeze(2)
    return lo + (hi - lo) * (t - idx)


def quantile_runs(models: list[nn.Module], fits: list[dict]) -> list[dict]:
    runs = []
    with torch.no_grad():
        for model, fit in zip(models, fits):
            raw = model(xva)  # (N, Q, H), as the head outputs it
            pred = raw.sort(dim=1).values  # the repair: quantiles in order
            runs.append(
                {
                    "samples": sample_from_quantiles(pred),
                    "median": pred[:, 4, :],
                    "fit": fit,
                    "raw": raw,
                    "pred": pred,
                }
            )
    return runs


def quantile_checks(runs: list[dict]) -> None:
    crossing = [
        float((r["raw"][:, 1:] < r["raw"][:, :-1]).float().mean()) for r in runs
    ]
    direct = [coverage(yva, r["pred"][:, 0], r["pred"][:, -1]) for r in runs]
    print(
        f"\nquantile crossing (q90 predicted below q80, etc.): {mean_sd(crossing)[0]:.2%} +/- {mean_sd(crossing)[1]:.2%} of all pairs"
    )
    print(
        f"coverage of [q10, q90] read straight off the head: {mean_sd(direct)[0]:.3f} +/- {mean_sd(direct)[1]:.3f}"
    )


class QuantileHead(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.trunk = GRUTrunk(hidden)
        self.out = nn.Linear(hidden, H * len(QUANTILES))

    def forward(self, x):
        return self.out(self.trunk(x)).reshape(-1, len(QUANTILES), H)


q_models, q_fits = fit_seeds(QuantileHead, lambda m, x, y: pinball(m(x), y))
q_runs = quantile_runs(q_models, q_fits)
report("GRU + quantiles", q_runs, n_params(q_models[0]))
quantile_checks(q_runs)

To put a quantile head in the same table, `sample_from_quantiles` turns its nine quantiles
into 100 sample paths: draw a level `u` between 0 and 1 and read the value off the
straight lines through the nine quantiles. Below q10 and above q90 there is no quantile to
read; the sampler extends the outermost segment with its own slope. That is an assumption
about the tails. The second check under the cell, the coverage of `[q10, q90]` read
straight off the head, is what the sampled 80% interval should come close to: 0.813 here,
against 0.797 sampled, the same order-statistics gap as for the Gaussian head.

**Quantile crossing** is the price of dropping the assumption. Nine independent linear
outputs have nothing forcing them into order, so sometimes the model claims a 90th
percentile below its own 80th — which is not a distribution at all. `sort()` patches it,
and everybody does that, but be clear that sorting is a repair applied after the fact,
not a model that could not produce the error.

## 1.7 The control: the same head on a linear model

`00_forecasting_baselines` measured a linear layer ahead of the GRU on this series
(validation MAE 0.2157 ± 0.0050 against 0.2400 ± 0.0091, three seeds). That result has to
be re-run here, or the rest of this notebook is measuring the wrong thing. Same pinball
loss, same nine quantiles, `nn.Linear` instead of a GRU.

In [ ]:
class LinearQuantile(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Linear(W, H * len(QUANTILES))

    def forward(self, x):
        return self.out(x).reshape(-1, len(QUANTILES), H)


lq_models, lq_fits = fit_seeds(LinearQuantile, lambda m, x, y: pinball(m(x), y))
lq_runs = quantile_runs(lq_models, lq_fits)
report("linear + quantiles", lq_runs, n_params(lq_models[0]))
quantile_checks(lq_runs)

Two things at once, and they pull in opposite directions.

**It wins, by more than the spread.** CRPS 0.1565 ± 0.0025 and MAE 0.2145 ± 0.0030, against
0.1783 ± 0.0051 and 0.2507 ± 0.0072 for `GRU + quantiles`, with the same loss and the same
nine quantiles. Both gaps (0.022 CRPS, 0.036 MAE) are four to five times the larger of the
two spreads. The ordering `00_forecasting_baselines` measured for MSE-trained point
forecasts survives the change of loss. It is not the smaller model that wins: the linear
head has 83,376 weights (192 inputs times 432 outputs, plus a bias each), the GRU head
40,944.

**It is the worse-calibrated head, and it crosses its own quantiles most.** Its `[q10, q90]`
covers 73.3% ± 1.5% of the truth, seven points short of 80, against 81.3% for the GRU
head. And it crosses on 12.9% ± 1.2% of pairs, almost three times the GRU version's
4.6% ± 1.5%. With no shared trunk and no nonlinearity, each of the 432 outputs (9
quantiles x 48 horizon steps) is its own independent regression on the same 192 inputs;
nothing couples them, so there is comparatively little stopping them from disagreeing
about order. The GRU version shares a trunk across all its outputs, which is presumably
why its crossing rate is lower without anything explicitly enforcing order there either.

Hold both facts. Picking by the metric alone would have hidden the second one entirely.

### Did every run converge?

A ranking means little if one arm was still learning when it was stopped. Below is the
validation loss of every run, each head on its own loss (the NLL can go below zero, the
pinball loss cannot, so each panel has its own scale), with the epoch early stopping kept
marked.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
heads = {
    "GRU + gaussian (NLL)": g_fits,
    "GRU + quantiles (pinball)": q_fits,
    "linear + quantiles (pinball)": lq_fits,
}
for ax, (name, fits) in zip(axes, heads.items()):
    for seed, fit in zip(SEEDS, fits):
        epochs = range(1, len(fit["curve"]) + 1)
        (line,) = ax.plot(epochs, fit["curve"], lw=1.2, label=f"seed {seed}")
        ax.plot(fit["best"], fit["curve"][fit["best"] - 1], "o", color=line.get_color())
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("epoch")
    ax.set_ylabel("validation loss")
axes[0].legend(fontsize=8)
fig.suptitle("validation loss per epoch; the dot is the epoch early stopping kept")
plt.tight_layout()

The pinball curves of both quantile heads flatten: the GRU's around 0.10 after six to nine
epochs, the linear head's around 0.085. The linear curves are still drifting down slowly
when the patience ends them (seed 0 stopped at 0.087, seeds 1 and 2 went on to about
0.085), and the GRU curves are flat for the five epochs after their best. So the linear
head's lead is not a GRU cut short; if anything the linear head is the one stopped a
little early. The Gaussian NLL curves jump by up to about 0.25 from one epoch to the
next, so which epoch counts as best is partly luck: the kept epochs are 5, 20 and 14.
That fits the Gaussian head's larger seed spread (0.0166 MAE, against 0.0072 for the GRU
quantile head). All of this holds for this budget, Adam at lr 1e-3 with a patience of 5;
a longer patience could change the noisy Gaussian runs most.

## 1.8 The table

Four columns, and they do not agree with each other. That is the useful part.

In [ ]:
print(
    f"{'model':20s} {'CRPS':>16} {'MAE':>16} {'cov80':>14} {'|80-cov|':>9} "
    f"{'params':>8} {'secs':>6}  best epochs"
)
print("-" * 112)
for name in sorted(scores, key=lambda k: scores[k]["crps"]):
    r = scores[name]
    best = "/".join(str(f["best"]) for f in r["fits"]) or "-"
    print(
        f"{name:20s} {r['crps']:7.4f} +/- {r['crps_sd']:.4f} {r['mae']:7.4f} +/- {r['mae_sd']:.4f} "
        f"{r['cov80']:6.3f} +/- {r['cov80_sd']:.3f} {abs(r['cov80'] - 0.8):9.3f} "
        f"{r['params']:8,} {r['secs']:6.1f}  {best}"
    )

Read the `+/-` columns first: each trained row is three seeds, and the spread says whether
two rows are ordered at all.

**A width alone buys a lot of CRPS.** The naive point forecast scored CRPS 0.7362 in 1.2;
`naive + residuals` starts from the same forecast, adds the spread of its own past errors,
and scores 0.4411, 40% lower. Its middle moved too (the median of the samples has MAE
0.6959 against 0.7362), but that is a small part of the drop.

**The free baseline is not competitive here, and the reason is its middle.** Every trained
model lands between 0.156 and 0.179 against the baseline's 0.44. That baseline is built on
plain naive, which `00_forecasting_baselines` already found to be the weak point forecast
on this series: a flat line through a day that rises and falls. Measured as point
forecasts, every trained median also beats the strong one: seasonal naive has a validation
MAE of 0.2906, the trained rows 0.2145 to 0.2508. Naming the baseline is the discipline;
which baseline turns out to be hard to beat is a property of the data.

**The linear model wins on CRPS and MAE; the two GRU heads tie.** `linear + quantiles`
(0.1565, 0.2145) is ahead of both GRU heads by four to five times their spread, and the
GRU heads are level with each other (CRPS 0.1783 and 0.1785, MAE 0.2507 and 0.2508). Next
to `00_forecasting_baselines` the point accuracy barely moves: the linear head's median
0.2145 ± 0.0030 against 00's linear 0.2157 ± 0.0050, the GRU quantile head's 0.2507 ±
0.0072 against 00's GRU 0.2400 ± 0.0091. Two things differ between those rows, not one: the
head, and the loss. 00 trained on MSE; the median here is trained with the pinball loss at
`q = 0.5`, which is MAE, the very metric of this column. Training on the metric itself
bought nothing measurable: both differences are within about one spread.

**The `|80-cov|` column ranks the models differently.** Both GRU heads cover the truth
with their 80% interval close to 80% of the time (`GRU + quantiles` 0.797, |80-cov| 0.003;
`GRU + gaussian` 0.791, 0.009), well within their seed spreads of 0.026 and 0.031. The CRPS
winner is the worst calibrated: `linear + quantiles` covers 72.0% (0.080), short on every
seed, just behind the free baseline, which is too wide (0.878, 0.078). Whether you care
depends entirely on what the forecast is for: CRPS if you are choosing a model,
calibration if somebody is going to size a threshold or a safety margin off your
intervals. Pick the metric from the decision, not from the leaderboard.

In [ ]:
picks = [40, 140, 240]
fig, axes = plt.subplots(1, len(picks), figsize=(14, 4.4), sharey=True)
runs = {
    "GRU + gaussian": g_runs,
    "GRU + quantiles": q_runs,
    "linear + quantiles": lq_runs,
}
shown = {name: r[0]["samples"] for name, r in runs.items()}  # seed 0 of each
colors = {
    "GRU + gaussian": "tab:blue",
    "GRU + quantiles": "tab:orange",
    "linear + quantiles": "tab:green",
}

for ax, i in zip(axes, picks):
    ax.plot(range(-2 * DAY, 0), xva[i, -2 * DAY :], color="0.6", lw=1)
    ax.plot(range(H), yva[i], color="k", lw=1.8, label="truth")
    for name, s in shown.items():
        lo, hi = s[i].quantile(0.1, dim=0), s[i].quantile(0.9, dim=0)
        ax.fill_between(range(H), lo, hi, alpha=0.18, color=colors[name])
        ax.plot(range(H), s[i].median(0).values, lw=1.2, color=colors[name], label=name)
    ax.axvline(0, color="0.85")
axes[0].set_ylabel("scaled demand")
axes[-1].legend(fontsize=7)
fig.suptitle("80% intervals on three validation windows (seed 0)")
plt.tight_layout()

for name, rs in runs.items():
    widths = [
        float(
            (
                r["samples"].quantile(0.9, dim=1) - r["samples"].quantile(0.1, dim=1)
            ).mean()
        )
        for r in rs
    ]
    m, sd = mean_sd(widths)
    print(
        f"{name:20s} mean 80% interval width over all validation windows {m:.3f} +/- {sd:.3f}"
    )

The medians are close to each other; the *widths* differ visibly. Averaged over all 361
validation windows, the linear band is the narrowest (0.603 ± 0.015, against 0.751 ± 0.043
for the GRU quantile head and 0.774 ± 0.107 for the Gaussian head), and it is the one that
covers the truth least often, the order of the coverage column. That is the whole notebook
in one picture: these models disagree more about how sure they are than about what will
happen, and only one of the two disagreements shows up in an MAE column.

## What to take from this

- A point forecast cannot answer a question about risk, and risk is usually the question.
- CRPS scores a distribution and reduces to MAE for a point forecast, so one table holds
  both. Report calibration next to it — CRPS alone will not tell you your 80% intervals
  cover the truth 72% of the time.
- A distributional head is a small change: same trunk, two outputs instead of one. On this
  series it cost the middle nothing measurable: the Gaussian head's mean and the quantile
  head's median land within about one seed spread of the MSE-trained GRU of
  `00_forecasting_baselines`.
- Gaussian assumes a shape and gets coherence for free; quantiles assume nothing and can
  cross. Neither is the right default — pick from what you know about the data.
- Reproduce the baseline inside every new framing. The linear model led in
  `00_forecasting_baselines` and, re-run here with the same pinball loss as the GRU, leads
  again on CRPS and MAE; re-running it is what tells you whether a new model beat the
  standing champion or a straw man.
- One run is an anecdote. Three seeds of the same Gaussian head differ by six points of
  80% coverage; a ranking or a calibration claim needs the spread next to it.
- The best score and the best calibration belong to different models in this table. Which
  one you ship is a question about the decision the forecast feeds.

## Exercises

**E1 — a stronger free baseline.** Build `seasonal naive + residuals`: repeat the last day,
`xtr[:, -DAY:]`, and resample its training residuals the way 1.3 does for plain naive.
Before running it, predict where it lands in the table of 1.8. Does any trained model
still beat it on CRPS, and on calibration?

**E2 — calibrate afterwards.** Take the Gaussian model's validation residuals, find the
constant `c` that makes `c * sigma` give exactly 80% coverage, and apply it to the test
split. Does a one-parameter recalibration hold up out of sample? This is conformal
prediction's core idea in three lines.

**E3 — fix the crossing.** Make `LinearQuantile` output the lowest quantile plus eight
non-negative increments (`softplus`), so the quantiles are monotone by construction.
Does forcing coherence cost accuracy?

**E4 — a fatter tail.** Swap the Gaussian head for a Student-t with a learned degrees-of-
freedom parameter (`torch.distributions.StudentT`). Where in the reliability curve does
it change things — the middle, or the 95% level?

**E5 — the honest question.** Every number in this notebook is validation. Re-score the
whole table on the test split. Given what `00_forecasting_baselines` 0.8 found on the same
test year (the ranking held, and the GRU's lead over seasonal naive shrank from 17% to 6%),
predict the answer before you run it: which rows move most, and does the CRPS ranking
survive?